# Tugas Besar 2 IF3170 Inteligensi Artifisial

## SVM Gaussian

Eksperimen `svm_gaussian` menggunakan struktur notebook kelompok dan preprocessing SVM lama.

---

## Group 17 (DerbyCounty)

- Juan Oloando Simanungkalit (13524032) - juanoloando.s@gmail.com
- Nashiruddin Akram (13524090) - akrambaasir@gmail.com
- Reinsen Silitonga (13524093) - reinsilitonga12@gmail.com
- Arghawisesa Dwinanda Arham (13524100) - arghaku.argha@gmail.com
- Manuel Thimoty Silalahi (13524102) - manuelsilalahi06@gmail.com

## Table of Contents

1. [Introduction](#1)
2. [Initialization](#2)
3. [Data Understanding](#3)
4. [Training and Validation Split](#4)
5. [Data Preparation](#5)
6. [Preparation Pipeline](#6)
7. [Modeling and Validation](#7)
8. [Comparison with scikit-learn](#8)
9. [Saving and Loading Models](#9)
10. [Kaggle Submission](#10)
11. [Error Analysis](#11)
12. [Bonus](#12)
13. [Referensi](#13)

---

# Introduction <a name="1"></a>

---

## Overview

Penginapan ingin memprediksi pembatalan pemesanan dari informasi tamu, tanggal kedatangan,
kanal pemesanan, harga, dan permintaan khusus. Target `is_canceled` bernilai 1 untuk pemesanan batal.

## Aim

Mengevaluasi SVM from scratch dan membandingkannya dengan scikit-learn pada pembagian data
dan preprocessing yang sama. Bagian model lain dibiarkan kosong karena notebook ini khusus SVM.

## Metric

Macro F1 merupakan rata-rata F1 kelas batal dan tidak batal. Threshold dipilih dari prediksi
out-of-fold pada data train. Nilai OOF yang juga dipakai memilih threshold bukan estimasi independen.

## Dataset

Input berada pada `data/train.csv`, `data/test.csv`, dan `data/sample_submission.csv`.
Train dan test diproses terpisah. Test tidak digunakan untuk memilih parameter atau threshold.

## Approach

Konfigurasi eksperimen: `{'kernel': 'gaussian', 'C': 1.0, 'gamma': 'scale'}`. Preprocessing, cleaning, dan fitur turunan didefinisikan langsung di notebook.
Susunan dokumentasi mengikuti template kelompok. Clipping, robust scaling, dan bobot KNN pada
template tidak ditambahkan ke eksperimen lama ini karena akan mengubah konfigurasi historis.

| Catatan historis | Nilai |
| --- | --- |
| Macro F1 OOF lama | 0.7907 |
| Threshold OOF lama | 0.43 |
| Public Kaggle | Belum ada submission teridentifikasi |

Angka historis berasal dari eksperimen sebelumnya dengan solver SMO. Kode training sekarang
menggunakan SLSQP dan kalibrasi BFGS. Evaluasi di bawah menghitung hasil eksekusi saat ini.
Public Kaggle tetap merujuk CSV lama yang telah dicocokkan dengan model tersimpan.

---

# Initialization <a name="2"></a>

---

## Environment Setup

Jalankan `pip install -r requirements.txt jupyter` dari root repo.
Seluruh grafik menggunakan Matplotlib dan disimpan sebagai PNG.
Perangkat CPU dipakai secara default agar tidak memerlukan PyTorch.

## Import Libraries

In [ ]:
import pickle
from scipy.optimize import minimize
from matplotlib.lines import Line2D
import hashlib
import inspect
import json
import random
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython import get_ipython
from IPython.display import Code, Markdown, display
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/train.csv').is_file())

if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')
plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Python', sys.version.split()[0], '| NumPy', np.__version__,
      '| pandas', pd.__version__, '| SciPy', scipy.__version__, '| sklearn', sklearn.__version__)

### Utilitas Data dan Validasi

Cleaning, metrik, dan validasi silang didefinisikan di notebook ini.

In [ ]:
TARGET = 'is_canceled'

ID_COL = 'id'

CAT_COLS = ['meal_package', 'room_category', 'booking_channel']

def clean_train(df):
    feature_cols = [c for c in df.columns if c != ID_COL]
    return df.drop_duplicates(subset=feature_cols).reset_index(drop=True)

def mark_invalid(df):
    df = df.copy()
    df.loc[df['num_adults'] == 0, 'num_adults'] = np.nan
    return df

def add_features(df):
    df = df.copy()
    df['total_nights'] = df['weekend_nights'] + df['weekday_nights']
    df['total_guests'] = df['num_adults'] + df['num_children']
    df['is_free'] = (df['nightly_rate_idr'] == 0).astype(float)
    df.loc[df['nightly_rate_idr'].isna(), 'is_free'] = np.nan
    df['rate_per_guest'] = df['nightly_rate_idr'] / df['total_guests'].replace(0, np.nan)

    arrival = pd.to_datetime(
        dict(year=df['arrival_year'], month=df['arrival_month'], day=df['arrival_day']),
        errors='coerce',
    )
    df['arrival_weekday'] = arrival.dt.weekday.astype(float)
    df['arrival_dayofyear'] = arrival.dt.dayofyear.astype(float)
    df['arrival_index'] = ((arrival - pd.Timestamp('2023-01-01')).dt.days).astype(float)
    return df

def save_model(bundle, path):
    # Simpan state agar pickle tidak bergantung pada modul model di luar notebook.
    bundle = dict(bundle)
    for key in ['model', 'preprocessor']:
        obj = bundle.get(key)
        if obj is not None:
            bundle[key + '_state'] = vars(obj).copy()
            bundle[key + '_class'] = type(obj).__name__
            del bundle[key]
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('wb') as file:
        pickle.dump(bundle, file)


def load_model(path):
    with Path(path).open('rb') as file:
        bundle = pickle.load(file)
    for key, default in [('model', 'SVM'), ('preprocessor', 'SVMPreprocessor')]:
        if key + '_state' in bundle:
            cls = globals().get(bundle.get(key + '_class', default))
            if cls is None:
                raise RuntimeError('Jalankan sel definisi kelas sebelum load model.')
            obj = cls.__new__(cls)
            obj.__dict__.update(bundle[key + '_state'])
            bundle[key] = obj
    return bundle


def confusion_matrix(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return np.array([[tn, fp], [fn, tp]])

def f1_per_class(y_true, y_pred):
    (tn, fp), (fn, tp) = confusion_matrix(y_true, y_pred)
    f1_pos = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn > 0 else 0.0
    f1_neg = 2 * tn / (2 * tn + fn + fp) if tn + fn + fp > 0 else 0.0
    return np.array([f1_neg, f1_pos])

def macro_f1(y_true, y_pred):
    return float(f1_per_class(y_true, y_pred).mean())

def accuracy(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))

def best_threshold(y_true, proba, grid=None):
    if grid is None:
        grid = np.round(np.arange(0.20, 0.81, 0.01), 2)
    scores = np.array([macro_f1(y_true, (proba >= t).astype(int)) for t in grid])
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])

def stratified_kfold(y, n_splits=5, seed=42):
    y = np.asarray(y)
    rng = np.random.RandomState(seed)
    fold_of = np.empty(len(y), dtype=int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        fold_of[idx] = np.arange(len(idx)) % n_splits
    return [(np.where(fold_of != k)[0], np.where(fold_of == k)[0]) for k in range(n_splits)]

def cross_val_proba(make_model, make_prep, df, y, folds):
    y = np.asarray(y)
    oof = np.zeros(len(y))
    for tr_idx, va_idx in folds:
        prep = make_prep()
        X_tr = prep.fit_transform(df.iloc[tr_idx])
        X_va = prep.transform(df.iloc[va_idx])
        model = make_model()
        model.fit(X_tr, y[tr_idx])
        oof[va_idx] = model.predict_proba(X_va)
    return oof

def fold_scores(y, proba, folds, threshold):
    y = np.asarray(y)
    pred = (proba >= threshold).astype(int)
    return np.array([macro_f1(y[va], pred[va]) for _, va in folds])

## Settings

In [ ]:
EXP_NAME = 'svm_gaussian'
PARAMS = {'kernel': 'gaussian', 'C': 1.0, 'gamma': 'scale'}
SEED = 42
N_FOLDS = 5
TEST_SIZE = 0.2
# CUDA hanya mempercepat kernel, solver SLSQP tetap berjalan pada CPU.
DEVICE = 'cpu'
SMOKE_TEST = False
RUN_SCREENING = False
COMMON_PARAMS = dict(tol=0.01, max_iter=1000, seed=SEED, device=DEVICE)
HISTORICAL_OOF = 0.7907188106522034
HISTORICAL_THRESHOLD = 0.43
PUBLIC_SCORE = None

DATA_DIR = ROOT / 'data'
RUN_DIR = ROOT / 'output/experiments' / EXP_NAME / ('smoke' if SMOKE_TEST else 'template_rerun')
FIG_DIR = RUN_DIR / 'figures'
MODEL_PATH = RUN_DIR / (EXP_NAME + '_rerun.pkl')
RERUN_SUB_PATH = RUN_DIR / ('submission_' + EXP_NAME + '_rerun.csv')
HISTORICAL_MODEL_PATH = ROOT / 'output/models' / (EXP_NAME + '.pkl')
HISTORICAL_CSV_PATH = ROOT / 'output/submissions' / ('submission_' + EXP_NAME + '.csv')
REPLAY_SUB_PATH = RUN_DIR / ('submission_' + EXP_NAME + '_historical.csv')
DEV_SIZE, HOLDOUT_SIZE, VISUAL_SIZE = (160, 80, 100) if SMOKE_TEST else (800, 400, 400)
PARTICLES, GENERATIONS = (3, 2) if SMOKE_TEST else (8, 4)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Mode:', 'uji kecil' if SMOKE_TEST else 'seluruh train', '| output:', RUN_DIR)

`SMOKE_TEST=True` membatasi train menjadi 320 baris dan test menjadi 12 baris.
Hasil uji kecil bukan skor full dataset dan tidak dipakai untuk memverifikasi submission historis.
`RUN_SCREENING` hanya digunakan pada eksperimen linear tuning. Output setiap eksperimen
disimpan terpisah sehingga model dan CSV submission lama tetap dapat digunakan sebagai pembanding.

## Seed Everything

In [ ]:
random.seed(SEED)
np.random.seed(SEED)

## Load Dataset

In [ ]:
train_raw = pd.read_csv(DATA_DIR / 'train.csv', dtype={'id': str})
test_raw = pd.read_csv(DATA_DIR / 'test.csv', dtype={'id': str})
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'id': str})
assert train_raw['id'].notna().all() and train_raw['id'].is_unique
assert test_raw['id'].notna().all() and test_raw['id'].is_unique
assert list(sample.columns) == ['id', TARGET] and test_raw['id'].equals(sample['id'])
assert set(train_raw[TARGET].unique()) == {0, 1}
print('Train:', train_raw.shape, '| test:', test_raw.shape)

---

# Data Understanding (Optional) <a name="3"></a>

---

## Describe Data

In [ ]:
display(pd.DataFrame({'dtype': train_raw.dtypes.astype(str),
                      'missing': train_raw.isna().sum(), 'unique': train_raw.nunique()}))
display(train_raw.describe().round(2))

## Explore Data

In [ ]:
counts = train_raw[TARGET].value_counts().sort_index()
ax = counts.plot.bar(color=['#246c9c', '#bf671f'], rot=0)
ax.set(title='Distribusi kelas pada train mentah', xlabel='is_canceled', ylabel='Jumlah pemesanan')
plt.tight_layout()
plt.savefig(FIG_DIR / 'class_balance.png', dpi=130)
plt.show()
display((counts / counts.sum()).rename('proporsi').to_frame().round(4))

#### Insights

Macro F1 memberi bobot yang sama kepada kedua kelas meskipun jumlah sampelnya berbeda.
Threshold OOF dan pembagian stratified digunakan untuk evaluasi. Resampling tidak diterapkan
pada eksperimen lama ini agar distribusi dan konfigurasi training tetap sesuai riwayatnya.

## Verify Data Quality

In [ ]:
duplicate_cols = [c for c in train_raw if c != 'id']
display(pd.Series({'duplikat fitur dan target': train_raw.duplicated(duplicate_cols).sum(),
                   'num_adults nol': train_raw['num_adults'].eq(0).sum(),
                   'baris dengan missing': train_raw.isna().any(axis=1).sum()}))

---

# Training and Validation Split <a name="4"></a>

---

Duplikat dengan fitur dan target identik dihapus sebelum pembagian data agar salinan baris
yang sama tidak tersebar ke train dan validasi. Holdout dipakai untuk pemeriksaan awal.
Evaluasi utama menggunakan lima fold pada train bersih dan tidak memakai label test.

In [ ]:
train_clean = clean_train(train_raw)
if SMOKE_TEST:
    selected, _ = train_test_split(np.arange(len(train_clean)), train_size=320,
                                  stratify=train_clean[TARGET], random_state=SEED)
    train_clean = train_clean.iloc[selected].reset_index(drop=True)
    test_raw = test_raw.head(12).copy()
    sample = sample.head(12).copy()
y = train_clean[TARGET].to_numpy()
train_set, val_set = train_test_split(train_clean, test_size=TEST_SIZE,
                                     stratify=y, random_state=SEED)
y_train, y_val = train_set[TARGET].to_numpy(), val_set[TARGET].to_numpy()
folds = stratified_kfold(y, n_splits=N_FOLDS, seed=SEED)
assert np.array_equal(np.sort(np.concatenate([va for _, va in folds])), np.arange(len(y)))
assert all(not np.intersect1d(tr, va).size for tr, va in folds)
print('Train bersih:', len(train_clean), '| train holdout:', len(train_set), '| validasi:', len(val_set))

---

# Data Preparation <a name="5"></a>

---

### Implementasi Preprocessing

In [ ]:
class BasePreprocessor:

    # feature_engineering: pakai add_features atau tidak; rare_min_count: batas kategori langka.
    def __init__(self, feature_engineering=True, rare_min_count=20, drop_cols=None):
        self.feature_engineering = feature_engineering
        self.rare_min_count = rare_min_count
        self.drop_cols = list(drop_cols or [])

    # Pelajari kolom dan kategori dari data train, lalu statistik khusus model.
    def fit(self, df):
        df = self._base(df)
        self.cat_cols_ = [c for c in CAT_COLS if c in df.columns]
        self.num_cols_ = [c for c in df.columns if c not in self.cat_cols_]
        self.keep_cats_ = {}
        for c in self.cat_cols_:
            counts = df[c].value_counts()
            self.keep_cats_[c] = sorted(counts[counts >= self.rare_min_count].index)
        self._fit_extra(self._group_rare(df))
        return self

    # Terapkan transformasi yang sudah dipelajari ke data validasi atau test.
    def transform(self, df):
        return self._transform_extra(self._group_rare(self._base(df)))

    # Gabungan fit lalu transform.
    def fit_transform(self, df):
        return self.fit(df).transform(df)

    # Statistik khusus model, default tidak ada.
    def _fit_extra(self, df):
        pass

    # Transformasi khusus model, default data dikembalikan apa adanya.
    def _transform_extra(self, df):
        return df

    # Tandai nilai janggal, tambah fitur turunan (opsional), buang kolom id dan target.
    def _base(self, df):
        df = mark_invalid(df)
        if self.feature_engineering:
            df = add_features(df)
        drop = [ID_COL, TARGET] + self.drop_cols
        return df.drop(columns=[c for c in drop if c in df.columns])

    # Ganti kategori yang tidak ada di daftar train menjadi 'Other'.
    def _group_rare(self, df):
        df = df.copy()
        for c in self.cat_cols_:
            known = df[c].isin(self.keep_cats_[c]) | df[c].isna()
            df.loc[~known, c] = 'Other'
        return df

class SVMPreprocessor(BasePreprocessor):
    # Pelajari statistik dari train untuk transform data validasi dan test.
    def _fit_extra(self, df):
        # Pelajari nilai imputasi, dengan fallback untuk kolom yang seluruhnya kosong.
        self.medians_ = df[self.num_cols_].median().fillna(0.0)
        self.modes_ = {c: df[c].mode().get(0, 'Other') for c in self.cat_cols_}

        # Simpan indikator missing dan susunan kolom hasil encoding.
        self.missing_cols_ = [c for c in df.columns if df[c].isna().any()]
        encoded = self._encode(df)
        self.columns_ = list(encoded.columns)

        # Simpan statistik scaling, std nol atau tidak tersedia diganti satu.
        self.means_ = encoded.mean()
        self.stds_ = encoded.std().fillna(1.0).replace(0, 1.0)

    # Pertahankan susunan fitur serta statistik yang dipelajari dari train.
    def _transform_extra(self, df):
        encoded = self._encode(df).reindex(columns=self.columns_, fill_value=0.0)
        return (encoded - self.means_) / self.stds_

    def _encode(self, df):
        out = pd.DataFrame(index=df.index)
        for c in self.missing_cols_:
            out[f'{c}_missing'] = df[c].isna().astype(float)
        out[self.num_cols_] = df[self.num_cols_].fillna(self.medians_)
        for c in self.cat_cols_:
            filled = df[c].fillna(self.modes_[c])
            for cat in dict.fromkeys(self.keep_cats_[c] + ['Other']):
                out[f'{c}_{cat}'] = (filled == cat).astype(float)
        return out.astype(float)

Seluruh transformasi berikut merupakan bagian dari kelas `SVMPreprocessor` di notebook ini.
Contoh pada bagian ini hanya memperlihatkan hasil proses. Pipeline training di bagian berikutnya
membuat preprocessor baru dan mempelajari statistik hanya dari data fit.

## Select Data

Semua fitur asli selain `id` dan target digunakan. `id` adalah pengenal pemesanan,
sedangkan memasukkan target ke fitur akan menyebabkan kebocoran label.
Belum ada eksperimen seleksi fitur terpisah pada konfigurasi historis ini.

In [ ]:
display(pd.Index([c for c in train_set.columns if c not in ['id', TARGET]], name='fitur asli'))

## Clean Data

### Missing Values

Fitur numerik diisi median train dan kategorikal diisi modus train. Kolom numerik yang
seluruhnya kosong memakai fallback 0. Indikator missing dibuat untuk kolom yang memiliki
missing pada data fit. Kategori langka atau baru dikelompokkan menjadi `Other`.

### Duplicates

`clean_train` menghapus duplikat fitur dan target pada train. Baris test tetap utuh.

### Outliers

Eksperimen ini tidak melakukan clipping atau menghapus outlier numerik.
Penambahan clipping akan menjadi eksperimen berbeda dari konfigurasi submission lama.

### Data Errors

Nilai `num_adults=0` ditandai sebagai missing sebelum imputasi.
Tanggal kedatangan yang tidak valid menghasilkan missing pada fitur tanggal.

In [ ]:
invalid_demo = mark_invalid(train_set)
display(pd.DataFrame({'missing sebelum': train_set.isna().sum(),
                      'missing setelah penandaan': invalid_demo.isna().sum()}))

## Construct Data

### Attribute Construction

Tujuh fitur turunan adalah `total_nights`, `total_guests`, `is_free`, `rate_per_guest`,
`arrival_weekday`, `arrival_dayofyear`, dan `arrival_index`. Perhitungan dilakukan per baris
sehingga tidak mempelajari informasi dari validasi atau test.

In [ ]:
feature_demo = add_features(invalid_demo)
constructed = [c for c in feature_demo if c not in train_set.columns]
display(feature_demo[constructed].head())

### Feature Scaling

Kolom hasil encoding distandardisasi dengan mean dan standar deviasi dari train.
Standar deviasi nol atau tidak tersedia diganti 1. Tidak ada bobot fitur khusus.

### Encoding

`meal_package`, `room_category`, dan `booking_channel` diubah menjadi one-hot.
Kategori dengan frekuensi di bawah 20 pada data fit dikelompokkan menjadi `Other`.
Susunan kolom validasi dan test mengikuti susunan kolom train.

### Dimensionality Reduction

PCA tidak digunakan pada model utama. Bonus visualisasi melatih model terpisah pada dua fitur,
sehingga gambar tersebut bukan proyeksi model yang dilatih pada semua fitur.

## Class Balance

Pembagian stratified mempertahankan proporsi kelas. Macro F1 dan threshold OOF digunakan
tanpa oversampling atau undersampling.

## Format Data

Keluaran preprocessing berupa matriks numerik finite dengan urutan kolom konsisten.

---

# Preparation Pipeline <a name="6"></a>

---

In [ ]:
prep_holdout = SVMPreprocessor()
X_train = prep_holdout.fit_transform(train_set)
X_val = prep_holdout.transform(val_set)
X_test_demo = prep_holdout.transform(test_raw)
assert X_train.columns.equals(X_val.columns) and X_train.columns.equals(X_test_demo.columns)
assert 'id' not in X_train and TARGET not in X_train
assert all(np.isfinite(frame.to_numpy()).all() for frame in [X_train, X_val, X_test_demo])
display(pd.Series({'fitur hasil preprocessing': X_train.shape[1],
                   'train holdout': len(X_train), 'validasi': len(X_val), 'test': len(X_test_demo)}))

---

# Modeling and Validation <a name="7"></a>

---

## K-Nearest Neighbors

## Decision Tree Learning

## Logistic Regression

## Support Vector Machine

SVM meminimalkan masalah dual dengan batas `0 <= alpha <= C` dan `sum(alpha * y) = 0`.
Linear memakai dot product, Polynomial memakai `(gamma * x.T @ z + coef0) ** degree`,
dan Gaussian memakai `exp(-gamma * ||x-z||²)`.

Optimasi blok SLSQP membatasi matriks kerja hingga 64 alpha. Kalibrasi sigmoid dengan BFGS
memetakan fungsi keputusan menjadi probabilitas menggunakan holdout internal dari data train.
Model utama tetap dilatih pada seluruh train fold.

In [ ]:
class SVM:

    # C mengatur penalti margin, tol toleransi optimasi, dan max_iter batas putaran training.
    def __init__(self, C=1.0, tol=1e-3, max_iter=1000, seed=42,
                 kernel='linear', degree=3, gamma='scale', coef0=0.0, device='cpu'):
        self.C = C
        self.tol = tol
        self.max_iter = max_iter
        self.seed = seed
        self.kernel = kernel
        self.degree = degree
        self.gamma = gamma
        self.coef0 = coef0
        self.device = device

    # Latih SVM, lalu pelajari kalibrasi probabilitas dari holdout di dalam data train.
    def fit(self, X, y):
        X = self._validate_X(X)
        y = np.asarray(y)
        if y.ndim != 1 or len(y) != len(X) or not np.isfinite(y).all():
            raise ValueError('y harus berupa vektor label numerik finite sepanjang X.')
        classes = np.unique(y)
        if len(classes) != 2:
            raise ValueError('SVM membutuhkan tepat dua kelas pada data train.')
        if not np.isfinite(self.C) or self.C <= 0:
            raise ValueError('C harus positif dan finite.')
        if not np.isfinite(self.tol) or self.tol <= 0:
            raise ValueError('tol harus positif dan finite.')
        if not isinstance(self.max_iter, (int, np.integer)) or self.max_iter < 1:
            raise ValueError('max_iter harus berupa bilangan bulat positif.')
        # Alias lama dipertahankan agar pemanggil dan model tersimpan tetap dapat digunakan.
        if self.kernel not in ('linear', 'poly', 'gaussian', 'rbf'):
            raise ValueError("kernel harus 'linear', 'poly', atau 'gaussian'.")
        if not isinstance(self.degree, (int, np.integer)) or self.degree < 1:
            raise ValueError('degree harus berupa bilangan bulat positif.')
        if not np.isfinite(self.coef0) or self.coef0 < 0:
            raise ValueError('coef0 harus nonnegatif dan finite.')
        if isinstance(self.gamma, str):
            if self.gamma not in ('scale', 'auto'):
                raise ValueError("gamma harus 'scale', 'auto', atau angka positif.")
        elif not np.isfinite(self.gamma) or self.gamma <= 0:
            raise ValueError('gamma harus positif dan finite.')

        self._fit_svm(X, y)
        self._calibrate(X, y)
        return self

    # Pecah QP dual menjadi blok kecil agar tidak menyimpan matriks kernel n x n.
    def _fit_svm(self, X, y):
        self.device_ = self._resolve_device()
        self.classes_, encoded = np.unique(y, return_inverse=True)
        self.n_features_in_ = X.shape[1]
        if self.gamma == 'scale':
            variance = X.var()
            self.gamma_ = 1.0 / (X.shape[1] * variance) if variance > 0 else 1.0
        elif self.gamma == 'auto':
            self.gamma_ = 1.0 / X.shape[1]
        else:
            self.gamma_ = float(self.gamma)

        signed_y = 2.0 * encoded - 1.0
        self.alpha_ = np.zeros(len(X))
        gradient = -np.ones(len(X))
        eps = 1e-8 * min(1.0, self.C)
        features = None
        if self.device_ == 'cuda':
            try:
                features = self._gpu_features(X)
            except MemoryError:
                if self.device != 'auto':
                    raise
                warnings.warn('Memori GPU tidak cukup; training beralih ke CPU.', RuntimeWarning)
                self.device_ = 'cpu'

        steps_per_pass = (len(X) + 63) // 64
        self.n_iter_ = 0
        for iteration in range(self.max_iter * steps_per_pass):
            # Dua himpunan kandidat memberi batas bawah dan atas bias pada kondisi KKT.
            values = -signed_y * gradient
            up = np.flatnonzero(((signed_y > 0) & (self.alpha_ < self.C - eps))
                                | ((signed_y < 0) & (self.alpha_ > eps)))
            low = np.flatnonzero(((signed_y > 0) & (self.alpha_ > eps))
                                 | ((signed_y < 0) & (self.alpha_ < self.C - eps)))
            if values[up].max() - values[low].min() <= 2.0 * self.tol:
                break
            working = np.unique(np.concatenate((up[np.argsort(values[up])[-32:]],
                                                low[np.argsort(values[low])[:32]])))
            if features is None:
                kernel = self._kernel(X, X[working])
            else:
                import torch
                try:
                    with torch.inference_mode():
                        kernel = self._kernel(features, features[working], xp=torch).cpu().numpy()
                except torch.cuda.OutOfMemoryError as error:
                    if self.device != 'auto':
                        raise MemoryError('VRAM tidak cukup untuk blok kernel SVM.') from error
                    warnings.warn('Memori GPU tidak cukup; training beralih ke CPU.', RuntimeWarning)
                    features = None
                    self.device_ = 'cpu'
                    kernel = self._kernel(X, X[working])
            columns = signed_y[:, None] * kernel * signed_y[working]
            block = columns[working]
            old = self.alpha_[working].copy()
            labels = signed_y[working]
            linear = gradient[working] - block @ old

            # SciPy menangani optimasi QP beserta batas alpha dan kendala jumlah alpha*y = 0.
            result = minimize(lambda a: (0.5 * a @ block @ a + linear @ a, block @ a + linear),
                              old, jac=True, method='SLSQP', bounds=[(0.0, self.C)] * len(old),
                              constraints={'type': 'eq', 'fun': lambda a: labels @ (a - old),
                                           'jac': lambda a: labels},
                              options={'ftol': min(1e-9, self.tol ** 2 * 1e-3), 'maxiter': 1000})
            if not result.success:
                raise RuntimeError(f'Solver QP SVM gagal: {result.message}')
            self.alpha_[working] = result.x
            gradient += columns @ (result.x - old)
            self.n_iter_ = iteration // steps_per_pass + 1

        values = -signed_y * gradient
        up = ((signed_y > 0) & (self.alpha_ < self.C - eps)) | ((signed_y < 0) & (self.alpha_ > eps))
        low = ((signed_y > 0) & (self.alpha_ > eps)) | ((signed_y < 0) & (self.alpha_ < self.C - eps))
        self.intercept_ = float((values[up].max() + values[low].min()) / 2.0)
        self.support_ = np.flatnonzero(self.alpha_ > eps)
        self.support_vectors_ = X[self.support_].copy()
        self.dual_coef_ = self.alpha_[self.support_] * signed_y[self.support_]
        self.coef_ = self.dual_coef_ @ self.support_vectors_ if self.kernel == 'linear' else None
        residual = signed_y * self.decision_function(X) - 1.0
        self.converged_ = not (((residual < -self.tol) & (self.alpha_ < self.C - eps))
                              | ((residual > self.tol) & (self.alpha_ > eps))).any()
        if not self.converged_:
            warnings.warn('SVM berhenti sebelum seluruh kondisi KKT memenuhi tol. '
                          'Periksa scaling atau naikkan max_iter.', RuntimeWarning, stacklevel=2)
        return self

    @staticmethod
    def _gpu_features(X, torch_device='cuda'):
        import torch
        try:
            return torch.as_tensor(X, dtype=torch.float64, device=torch_device)
        except torch.cuda.OutOfMemoryError as error:
            raise MemoryError('VRAM tidak cukup untuk data SVM.') from error

    # PyTorch opsional: mode CPU dan prediksi model tersimpan tidak mengimpornya.
    def _resolve_device(self):
        device = getattr(self, 'device', 'cpu')
        if device not in ('cpu', 'cuda', 'auto'):
            raise ValueError("device harus 'cpu', 'cuda', atau 'auto'.")
        if device == 'cpu':
            return 'cpu'
        try:
            import torch
        except ImportError as error:
            if device == 'auto':
                return 'cpu'
            raise RuntimeError('device=cuda memerlukan PyTorch dengan dukungan CUDA.') from error
        if torch.cuda.is_available():
            return 'cuda'
        if device == 'auto':
            return 'cpu'
        raise RuntimeError('CUDA tidak tersedia. Aktifkan GPU dan gunakan PyTorch CUDA, atau device=cpu.')

    # Hitung kemiripan seluruh pasangan baris pada X dan Y.
    def _kernel(self, X, Y, xp=np):
        with np.errstate(over='ignore', invalid='ignore'):
            product = X @ Y.T
            if self.kernel == 'poly':
                result = (self.gamma_ * product + self.coef0) ** self.degree
            elif self.kernel in ('gaussian', 'rbf'):
                distance = (X ** 2).sum(axis=1)[:, None] + (Y ** 2).sum(axis=1) - 2.0 * product
                result = xp.exp(-self.gamma_ * distance.clip(0.0, None))
            else:
                result = product
        if not xp.isfinite(result).all():
            raise ValueError('Perhitungan kernel tidak finite, periksa scaling dan parameter kernel.')
        return result

    # Gunakan 20 persen dari tiap kelas untuk kalibrasi model sementara.
    def _calibrate(self, X, y):
        positive = y == self.classes_[1]
        counts = np.bincount(positive.astype(int), minlength=2)
        if counts.min() < 2:
            self.probA_ = 0.0
            self.probB_ = float(np.log((counts[0] + 1.0) / (counts[1] + 1.0)))
            warnings.warn('Sampel tiap kelas terlalu sedikit untuk holdout kalibrasi. '
                          'Probabilitas memakai prior kelas.', RuntimeWarning, stacklevel=2)
            return

        rng = np.random.default_rng(self.seed)
        holdout = []
        for label in self.classes_:
            indices = rng.permutation(np.flatnonzero(y == label))
            size = max(1, len(indices) // 5)
            holdout.extend(indices[:size])
        holdout = np.asarray(holdout)
        training = np.ones(len(X), dtype=bool)
        training[holdout] = False

        calibration_model = SVM(C=self.C, tol=self.tol, max_iter=self.max_iter,
                                seed=self.seed, kernel=self.kernel, degree=self.degree,
                                gamma=self.gamma_, coef0=self.coef0, device=self.device_)
        calibration_model._fit_svm(X[training], y[training])
        scores = calibration_model.decision_function(X[holdout])
        self._fit_sigmoid(scores, positive[holdout])

    # Pelajari sigmoid P(kelas kedua) = 1 / (1 + exp(A * score + B)).
    def _fit_sigmoid(self, scores, positive):
        n_positive = int(positive.sum())
        n_negative = len(positive) - n_positive
        target = np.where(positive, (n_positive + 1.0) / (n_positive + 2.0),
                          1.0 / (n_negative + 2.0))
        center = scores.mean()
        scale = scores.std()
        if scale == 0:
            scale = 1.0
        design = np.column_stack(((scores - center) / scale, np.ones(len(scores))))
        params = np.array([0.0, np.log((n_negative + 1.0) / (n_positive + 1.0))])

        def objective(params):
            logits = design @ params
            probability = np.exp(-np.logaddexp(0.0, logits))
            loss = np.sum(np.logaddexp(0.0, -logits) + target * logits)
            return loss, design.T @ (target - probability)

        result = minimize(objective, params, jac=True, method='BFGS',
                          options={'gtol': 1e-5, 'maxiter': 100})
        if not result.success and np.max(np.abs(result.jac)) > 1e-5:
            warnings.warn(f'Kalibrasi sigmoid belum konvergen: {result.message}', RuntimeWarning)
        params = result.x
        self.probA_ = float(params[0] / scale)
        self.probB_ = float(params[1] - self.probA_ * center)

    # Pastikan fitur berupa matriks numerik tanpa NaN atau infinity.
    @staticmethod
    def _validate_X(X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] == 0 or not np.isfinite(X).all():
            raise ValueError('X harus berupa matriks numerik finite dengan minimal satu fitur.')
        return X

    # Kernel linear memakai w.x + b, kernel lainnya memakai kontribusi support vector.
    def decision_function(self, X):
        if not hasattr(self, 'support_'):
            raise RuntimeError('SVM harus di-fit sebelum melakukan prediksi.')
        X = self._validate_X(X)
        if X.shape[1] != self.n_features_in_:
            raise ValueError('Jumlah fitur harus sama dengan data train.')
        if getattr(self, 'kernel', 'linear') == 'linear':
            return X @ self.coef_ + self.intercept_
        scores = np.empty(len(X))
        for start in range(0, len(X), 256):
            kernel = self._kernel(X[start:start + 256], self.support_vectors_)
            scores[start:start + 256] = kernel @ self.dual_coef_ + self.intercept_
        return scores

    # Probabilitas kelas kedua dalam array satu dimensi untuk pipeline validasi tim.
    def predict_proba(self, X):
        if not hasattr(self, 'probA_'):
            raise RuntimeError('SVM harus di-fit dengan kalibrasi sebelum menghitung probabilitas.')
        logits = self.probA_ * self.decision_function(X) + self.probB_
        return np.exp(-np.logaddexp(0.0, logits))

    # Gunakan tanda score secara default, atau threshold probabilitas jika diberikan.
    def predict(self, X, threshold=None):
        if threshold is not None:
            if not np.isfinite(threshold) or not 0 <= threshold <= 1:
                raise ValueError('threshold harus berada dalam rentang 0 sampai 1.')
            probability = self.predict_proba(X)
            return self.classes_[(probability >= threshold).astype(int)]
        scores = self.decision_function(X)
        return self.classes_[(scores >= 0).astype(int)]

### Visualisasi SVM

Matplotlib menampilkan pemisah, margin, support vector, dan hasil evaluasi.

In [ ]:
def show_plot(fig, path):
    from IPython.display import display

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=130, bbox_inches='tight')
    display(fig)
    plt.close(fig)

def boundary_plot(X, y, models, names):
    lower, upper = X.min(axis=0) - 0.5, X.max(axis=0) + 0.5
    xx, yy = np.meshgrid(np.linspace(lower[0], upper[0], 160),
                         np.linspace(lower[1], upper[1], 160))
    grid = np.column_stack((xx.ravel(), yy.ravel()))
    fig, axes = plt.subplots(1, len(models), figsize=(13, 4), sharex=True, sharey=True, squeeze=False)
    for ax, (kernel, model) in zip(axes[0], models.items()):
        score = model.decision_function(grid).reshape(xx.shape)
        ax.contourf(xx, yy, (score >= 0).astype(int), levels=[-0.5, 0.5, 1.5],
                    colors=['tab:blue', 'tab:orange'], alpha=0.12)
        ax.contour(xx, yy, score, levels=[-1, 0, 1], colors='black',
                   linestyles=['--', '-', '--'], linewidths=[1, 1.5, 1])
        for label, color, marker, text in [(0, 'tab:blue', 'o', 'Tidak batal'),
                                           (1, 'tab:orange', '^', 'Batal')]:
            mask = y == label
            ax.scatter(X[mask, 0], X[mask, 1], color=color, marker=marker,
                       s=16, alpha=0.65, label=text)
        sv = model.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=55, facecolors='none',
                   edgecolors='black', linewidths=0.7, label='Support vector')
        ax.set(title=f'{names[kernel]} ({len(sv)} SV)', xlabel='Jeda kedatangan (z-score)')
    axes[0, 0].set_ylabel('Tarif kamar (z-score)')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    handles += [Line2D([], [], color='black'), Line2D([], [], color='black', linestyle='--')]
    labels += ['Pemisah f=0', 'Margin f=±1']
    fig.legend(handles, labels, loc='lower center', ncol=5)
    fig.suptitle(f'Pemisah, margin, dan support vector SVM pada {len(X)} baris train')
    fig.tight_layout(rect=[0, 0.12, 1, 0.93])
    return fig

def pso_plot(history, comparison, dev_size, holdout_size):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
    axes[0].plot(range(1, len(history) + 1), history, 'o-', label='Fitness terbaik')
    axes[0].axhline(comparison.iloc[0]['oof_macro_f1'], color='gray', linestyle='--', label='Baseline')
    axes[0].set(title=f'Pencarian PSO pada {dev_size} baris', xlabel='Generasi', ylabel='Macro F1 OOF')
    axes[0].legend()
    comparison[['oof_macro_f1', 'holdout_macro_f1']].plot.bar(ax=axes[1], rot=0)
    axes[1].set(title='Baseline dan hasil PSO', xlabel='', ylabel='Macro F1', ylim=(0, 1))
    axes[1].legend(['OOF seleksi', f'Holdout ({holdout_size} baris)'])
    return fig

def kernel_plot(results, details, names, rows):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
    table = results.pivot(index='kernel', columns='implementation', values='oof_macro_f1')
    table = table.reindex(list(names)).rename(index=names)
    table.rename(columns={'from_scratch': 'From scratch', 'sklearn': 'scikit-learn'}).plot.bar(ax=axes[0], rot=0)
    axes[0].set(title=f'OOF pada {rows:,} baris', xlabel='Kernel', ylabel='Macro F1', ylim=(0, 1))
    for kernel, label in names.items():
        part = details[(details['kernel'] == kernel) & (details['implementation'] == 'from_scratch')]
        axes[1].plot(part['fold'], part['macro_f1'], marker='o', label=label)
    axes[1].set(title='Skor from scratch tiap fold', xlabel='Fold', ylabel='Macro F1')
    axes[1].legend()
    return fig

In [ ]:
# SLSQP menyelesaikan QP dual per blok, sedangkan BFGS mempelajari sigmoid kalibrasi.
# Kernel, working set, bias, support vector, dan prediksi dihitung oleh kelas SVM.
display(Code(inspect.getsource(SVM._kernel), language='python'))
with threadpool_limits(limits=1):
    holdout_model = SVM(**COMMON_PARAMS, **PARAMS).fit(X_train, y_train)
    holdout_pred = holdout_model.predict(X_val)
display(pd.Series({'macro_f1_holdout_decision_0': macro_f1(y_val, holdout_pred),
                   'konvergen': holdout_model.converged_}))

## Naive Bayes

## Model Evaluation

Validasi awal memakai tanda fungsi keputusan SVM tanpa tuning threshold.
Evaluasi utama berikut memakai probabilitas OOF untuk menentukan threshold Macro F1.
Kedua prosedur prediksi ini berbeda dan skornya tidak diperlakukan sebagai perbandingan langsung.

## Improvements (Optional)

Setiap fold membuat preprocessor baru. Hasil konvergensi dan peringatan dicatat agar model
yang bermasalah terlihat pada analisis. Skor lama tidak ditempelkan sebagai output run baru.

In [ ]:
scratch_models = []
def make_scratch():
    model = SVM(**COMMON_PARAMS, **PARAMS)
    scratch_models.append(model)
    return model

started = time.perf_counter()
with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as scratch_messages:
    warnings.simplefilter('always', RuntimeWarning)
    oof_scratch = cross_val_proba(make_scratch, SVMPreprocessor, train_clean, y, folds)
scratch_seconds = time.perf_counter() - started
assert oof_scratch.shape == y.shape and np.isfinite(oof_scratch).all()
assert ((oof_scratch >= 0) & (oof_scratch <= 1)).all()
threshold, scratch_score = best_threshold(y, oof_scratch)
assert np.isclose(scratch_score, f1_score(y, oof_scratch >= threshold, average='macro'))
scratch_fold_scores = fold_scores(y, oof_scratch, folds, threshold)
fold_diagnostics = pd.DataFrame({
    'fold': np.arange(1, N_FOLDS + 1), 'macro_f1': scratch_fold_scores,
    'iterations': [m.n_iter_ for m in scratch_models],
    'converged': [bool(m.converged_) for m in scratch_models],
})
display(fold_diagnostics.round(4))
print('OOF saat ini:', round(scratch_score, 4), '| threshold:', threshold,
      '| peringatan:', len(scratch_messages), '| detik:', round(scratch_seconds, 1))
for message in sorted({str(w.message) for w in scratch_messages}):
    print(message)

---

# Comparison with scikit-learn <a name="8"></a>

---

Parameter kernel, preprocessing, dan lima fold luar disamakan.
Kalibrasi from scratch menggunakan holdout internal 20 persen, sedangkan pembanding memakai
lima fold internal dengan `ensemble=False`. Perbedaan kalibrasi ini dapat memengaruhi skor
probabilitas dan threshold, sehingga selisih Macro F1 bukan hanya akibat solver SVM.

In [ ]:
class SklearnWrapper:
    def __init__(self):
        reference_params = dict(PARAMS)
        if reference_params['kernel'] == 'gaussian':
            reference_params['kernel'] = 'rbf'
        self.model = CalibratedClassifierCV(
            SVC(**reference_params, tol=COMMON_PARAMS['tol'], random_state=SEED),
            method='sigmoid', ensemble=False,
            cv=StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED))

    def fit(self, X, labels):
        self.model.fit(X, labels)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]

started = time.perf_counter()
with threadpool_limits(limits=1):
    oof_sklearn = cross_val_proba(SklearnWrapper, SVMPreprocessor, train_clean, y, folds)
sklearn_seconds = time.perf_counter() - started
reference_threshold, reference_score = best_threshold(y, oof_sklearn)
reference_fold_scores = fold_scores(y, oof_sklearn, folds, reference_threshold)
comparison = pd.DataFrame([
    dict(model='From scratch', macro_f1=scratch_score, threshold=threshold,
         fold_std=float(scratch_fold_scores.std()), seconds=scratch_seconds),
    dict(model='scikit-learn', macro_f1=reference_score, threshold=reference_threshold,
         fold_std=float(reference_fold_scores.std()), seconds=sklearn_seconds),
]).set_index('model')
display(comparison.round(4))
comparison.to_csv(RUN_DIR / 'comparison.csv')
np.savez_compressed(RUN_DIR / 'oof.npz', y=y, scratch=oof_scratch, sklearn=oof_sklearn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
comparison['macro_f1'].plot.bar(ax=axes[0], color=['#246c9c', '#bf671f'], rot=0)
axes[0].set(title='Macro F1 OOF pada data yang sama', ylabel='Macro F1', ylim=(0, 1))
for i, value in enumerate(comparison['macro_f1']):
    axes[0].text(i, value + 0.02, f'{value:.4f}', ha='center')
axes[1].plot(range(1, N_FOLDS + 1), scratch_fold_scores, 'o-', label='From scratch', color='#246c9c')
axes[1].plot(range(1, N_FOLDS + 1), reference_fold_scores, 's--', label='scikit-learn', color='#bf671f')
axes[1].set(title='Skor tiap fold', xlabel='Fold', ylabel='Macro F1', xticks=range(1, N_FOLDS + 1))
axes[1].legend()
fig.suptitle(f'{EXP_NAME} pada {len(y):,} baris train bersih')
fig.savefig(FIG_DIR / 'comparison.png', dpi=130)
plt.show()
display(Markdown(f"Macro F1 OOF from scratch sebesar **{scratch_score:.4f}**, "
                 f"sedangkan scikit-learn sebesar **{reference_score:.4f}**. "
                 f"Selisihnya {scratch_score - reference_score:+.4f}. "
                 "Std pada tabel menunjukkan penyebaran skor antar-fold, bukan confidence interval."))

---

# Saving and Loading Models <a name="9"></a>

---

Preprocessor baru dan SVM dilatih pada seluruh train bersih yang dipakai dalam run ini.
Bundle berisi model, preprocessor, threshold OOF saat ini, jumlah baris, dan hash input.
Model historis tidak ditimpa. Hasil load diperiksa terhadap prediksi sebelum penyimpanan.

In [ ]:
final_prep = SVMPreprocessor()
final_X_train = final_prep.fit_transform(train_clean)
final_X_test = final_prep.transform(test_raw)
assert final_X_train.columns.equals(final_X_test.columns)
with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as final_messages:
    warnings.simplefilter('always', RuntimeWarning)
    final_model = SVM(**COMMON_PARAMS, **PARAMS).fit(final_X_train, y)
    probability_before = final_model.predict_proba(final_X_test)
if not final_model.converged_ or final_messages:
    raise RuntimeError('Training final belum bersih: ' + ' | '.join(str(w.message) for w in final_messages))
save_model(dict(model=final_model, preprocessor=final_prep, threshold=threshold,
                train_rows=len(train_clean), smoke_test=SMOKE_TEST,
                train_sha256=hashlib.sha256((DATA_DIR / 'train.csv').read_bytes()).hexdigest()), MODEL_PATH)
loaded_bundle = load_model(MODEL_PATH)
with threadpool_limits(limits=1):
    loaded_probability = loaded_bundle['model'].predict_proba(loaded_bundle['preprocessor'].transform(test_raw))
np.testing.assert_array_equal(probability_before, loaded_probability)
print('Model run ini:', MODEL_PATH)

---

# Kaggle Submission <a name="10"></a>

---

CSV training ulang menggunakan threshold OOF yang dihitung pada run ini.
Untuk baseline dan polynomial, CSV historis juga direproduksi dari model lama tersimpan
dan dibandingkan baris demi baris dengan file yang diunduh dari Kaggle.
Public score historis hanya berlaku untuk CSV reproduksi tersebut.
Ekspor ini tidak mengirim submission otomatis ke Kaggle.

In [ ]:
rerun_pred = (loaded_probability >= loaded_bundle['threshold']).astype(int)
submission = pd.DataFrame({'id': test_raw['id'], TARGET: rerun_pred})
assert submission.shape == sample.shape and submission.columns.equals(sample.columns)
assert submission['id'].equals(sample['id']) and submission[TARGET].isin([0, 1]).all()
submission.to_csv(RERUN_SUB_PATH, index=False)
pd.testing.assert_frame_equal(submission, pd.read_csv(RERUN_SUB_PATH, dtype={'id': str}))
print('CSV training ulang:', RERUN_SUB_PATH)

replay_verified = False
if PUBLIC_SCORE is not None and not SMOKE_TEST:
    historical_bundle = load_model(HISTORICAL_MODEL_PATH)
    with threadpool_limits(limits=1):
        historical_pred = historical_bundle['model'].predict(
            historical_bundle['preprocessor'].transform(test_raw),
            threshold=historical_bundle['threshold'])
    replay = pd.DataFrame({'id': test_raw['id'], TARGET: historical_pred})
    expected = pd.read_csv(HISTORICAL_CSV_PATH, dtype={'id': str})
    pd.testing.assert_frame_equal(replay, expected)
    replay.to_csv(REPLAY_SUB_PATH, index=False)
    replay_verified = True
    differences = int((rerun_pred != historical_pred).sum())
    print('Reproduksi CSV Kaggle cocok:', len(replay), 'baris')
    print('Public score CSV historis:', PUBLIC_SCORE)
    print('Perbedaan training ulang terhadap historis:', differences, 'prediksi')
    print('CSV historis:', REPLAY_SUB_PATH)
elif PUBLIC_SCORE is None:
    print('Eksperimen pendukung tanpa public score Kaggle teridentifikasi.')
else:
    print('Mode uji kecil tidak memverifikasi CSV submission historis.')
display(submission.head())

---

# Error Analysis <a name="11"></a>

---

## Confusion Matrix

Analisis menggunakan prediksi OOF from scratch dengan threshold terpilih.
FP berarti pemesanan sebenarnya tidak batal tetapi diprediksi batal.
FN berarti pemesanan sebenarnya batal tetapi diprediksi tidak batal.

In [ ]:
oof_pred = (oof_scratch >= threshold).astype(int)
matrix = confusion_matrix(y, oof_pred)
fig, ax = plt.subplots(figsize=(5, 4), layout='constrained')
image = ax.imshow(matrix, cmap='Blues')
for (row, col), value in np.ndenumerate(matrix):
    ax.text(col, row, str(value), ha='center', va='center',
            color='white' if value > matrix.max() / 2 else 'black')
ax.set(title='Confusion matrix OOF', xlabel='Prediksi', ylabel='Aktual',
       xticks=[0, 1], yticks=[0, 1], xticklabels=['Tidak batal', 'Batal'],
       yticklabels=['Tidak batal', 'Batal'])
fig.colorbar(image, ax=ax, label='Jumlah pemesanan')
fig.savefig(FIG_DIR / 'confusion_matrix.png', dpi=130)
plt.show()
display(pd.Series(f1_per_class(y, oof_pred), index=['F1 tidak batal', 'F1 batal']).round(4))

## Kesalahan per Kelompok

In [ ]:
errors = train_clean[['booking_channel', 'special_requests']].copy()
errors['salah'] = (oof_pred != y).astype(int)
error_groups = errors.groupby('booking_channel', dropna=False)['salah'].agg(['count', 'mean'])
error_groups.columns = ['jumlah', 'proporsi_salah']
display(error_groups.sort_values('proporsi_salah', ascending=False).round(4))
display(errors.groupby('special_requests', dropna=False)['salah'].agg(['count', 'mean']).round(4))
tn, fp, fn, tp = matrix.ravel()
display(Markdown(f"Kesalahan OOF terdiri dari **{fp} FP** dan **{fn} FN**. "
                 "Proporsi salah per kelompok harus dibaca bersama jumlah sampelnya. "
                 "Kelompok berukuran kecil belum cukup untuk menyimpulkan pola umum."))

---

# Bonus <a name="12"></a>

---

## Visualisasi dua fitur

Setiap kernel dilatih ulang menggunakan dua fitur untuk menggambarkan batas
keputusannya. Grafik ini tidak memproyeksikan model yang dilatih pada seluruh fitur.

Garis penuh menunjukkan $f(x)=0$, sedangkan garis putus-putus menunjukkan
$f(x)=\pm1$. Lingkaran menandai support vector. Pada kernel non-linear, jarak
antarkontur dalam gambar tidak menyatakan lebar margin Euclidean di ruang asal.
Support vector pada SVM soft-margin juga dapat berada di dalam margin.

Bagian bonus menggunakan variabel terpisah agar model, threshold, dan CSV eksperimen utama
tidak berubah. Setiap kernel ditampilkan pada dua fitur yang sama. Hasil PSO merupakan
eksperimen Gaussian tambahan dan tidak menggantikan konfigurasi historis notebook ini.

In [ ]:
def prepare_bonus():
    # Siapkan sampel bonus terpisah dari populasi evaluasi utama.
    # PSO hanya memakai data pengembangan. Holdout dievaluasi setelah pencarian selesai.
    train = train_clean.copy()
    sample_idx, _ = train_test_split(np.arange(len(train)), train_size=DEV_SIZE + HOLDOUT_SIZE,
                                     stratify=train[TARGET], random_state=SEED)
    dev_idx, holdout_idx = train_test_split(sample_idx, test_size=HOLDOUT_SIZE,
                                           stratify=train.iloc[sample_idx][TARGET], random_state=SEED)
    assert not np.intersect1d(dev_idx, holdout_idx).size
    dev = train.iloc[dev_idx].reset_index(drop=True)
    holdout = train.iloc[holdout_idx].reset_index(drop=True)
    y_dev = dev[TARGET].to_numpy()
    folds = stratified_kfold(y_dev, n_splits=5, seed=SEED)
    assert np.array_equal(np.sort(np.concatenate([va for _, va in folds])), np.arange(DEV_SIZE))
    # Fit preprocessing sekali per train fold dan pakai ulang transform yang sama untuk semua kandidat.
    prepared_folds = []
    for tr, va in folds:
        assert not np.intersect1d(tr, va).size
        prep = SVMPreprocessor()
        prepared_folds.append((tr, va, prep.fit_transform(dev.iloc[tr]).to_numpy(),
                               prep.transform(dev.iloc[va]).to_numpy()))
    display(pd.DataFrame({'bagian': ['Pengembangan', 'Holdout'],
                          'baris': [len(dev), len(holdout)],
                          'proporsi_batal': [y_dev.mean(), holdout[TARGET].mean()]}).round(4))
    return dev, holdout, y_dev, folds, prepared_folds

In [ ]:
def visualize_bonus(dev, y_dev):
    # Gunakan waktu tunggu dan tarif kamar untuk memperlihatkan bentuk batas keputusan.
    visual_idx, _ = train_test_split(np.arange(len(dev)), train_size=VISUAL_SIZE,
                                     stratify=y_dev, random_state=SEED)
    features_2d = ['days_before_arrival', 'nightly_rate_idr']
    raw_2d = dev.iloc[visual_idx][features_2d]
    # Hitung median dan statistik standardisasi dari sampel visualisasi. Tarif nol tetap disertakan.
    filled_2d = raw_2d.fillna(raw_2d.median())
    X_2d = ((filled_2d - filled_2d.mean()) / filled_2d.std(ddof=0).replace(0, 1)).to_numpy()
    y_2d = y_dev[visual_idx]
    assert X_2d.shape == (VISUAL_SIZE, 2) and np.isfinite(X_2d).all()
    # Latih tiga model 2D pada sampel yang sama agar bentuk batasnya dapat dibandingkan.
    visual_models = {}
    with threadpool_limits(limits=1):
        for kernel in ['linear', 'poly', 'gaussian']:
            model = SVM(kernel=kernel, C=1, degree=3, coef0=1, gamma='scale',
                        tol=0.01, max_iter=1000, seed=SEED, device=DEVICE).fit(X_2d, y_2d)
            assert model.converged_
            visual_models[kernel] = model
        fig = boundary_plot(X_2d, y_2d, visual_models,
                            {'linear': 'Linear', 'poly': 'Polynomial', 'gaussian': 'Gaussian'})
    visual_results = [dict(kernel=k, support_vectors=len(m.support_), converged=bool(m.converged_))
                      for k, m in visual_models.items()]
    show_plot(fig, FIG_DIR / (EXP_NAME + '_boundaries.png'))
    return visual_results

In [ ]:
bonus_dev, bonus_holdout, bonus_y, bonus_folds, bonus_prepared = prepare_bonus()
bonus_visual = visualize_bonus(bonus_dev, bonus_y)

## Optimasi PSO

Setiap partikel mewakili kandidat hyperparameter $x=(\log_{10}C,\log_{10}g)$ dengan
$\gamma=g/(d\,\mathrm{Var}(X_{train}))$. Posisi kandidat diperbarui berdasarkan posisi terbaik
partikel $p_i$ dan posisi terbaik seluruh swarm $g^*$:

$$v_i\leftarrow wv_i+c_1r_1(p_i-x_i)+c_2r_2(g^*-x_i),\qquad x_i\leftarrow x_i+v_i.$$

Fitness kandidat dihitung dari Macro F1 OOF setelah pemilihan threshold. PSO
mencari hyperparameter, sementara SLSQP melatih SVM pada setiap kandidat. Nilai
fitness terbaik dipertahankan selama pencarian, tetapi peningkatan pada holdout
dan pencapaian optimum global tetap tidak terjamin.

Implementasi PSO mengacu pada referensi [1] dan [2]. Penerapannya untuk tuning SVM
mengacu pada referensi [3]. Eksperimen ini tidak menerapkan feature selection yang
dibahas dalam penelitian tersebut.

In [ ]:
# PSO dengan inertia weight: Kennedy & Eberhart (1995), doi:10.1109/ICNN.1995.488968.
# Shi & Eberhart (1998), doi:10.1109/ICEC.1998.699146.
def particle_swarm(objective, bounds, initial, particles=8, generations=4, seed=42):
    bounds = np.asarray(bounds, dtype=float)
    initial = np.asarray(initial, dtype=float)
    if (bounds.shape != (len(initial), 2) or not np.isfinite(bounds).all()
            or not np.isfinite(initial).all() or np.any(bounds[:, 0] >= bounds[:, 1])
            or np.any(initial < bounds[:, 0]) or np.any(initial > bounds[:, 1])
            or particles < 2 or generations < 1):
        raise ValueError('Batas, posisi awal, dan anggaran PSO tidak valid.')
    rng = np.random.default_rng(seed)
    lower, upper = bounds.T
    positions = rng.uniform(lower, upper, size=(particles, len(initial)))
    # Sertakan baseline sebagai satu posisi awal, lalu simpan pengalaman terbaik tiap partikel.
    positions[0] = initial
    velocity = np.zeros_like(positions)
    personal = positions.copy()
    personal_scores = np.full(particles, -np.inf)
    trace = []
    for generation in range(generations):
        scores = np.array([objective(position.copy()) for position in positions])
        if not np.isfinite(scores).all():
            raise ValueError('Fitness PSO harus finite.')
        # Perbarui personal best dan global best berdasarkan fitness yang benar-benar dievaluasi.
        improved = scores > personal_scores
        personal[improved], personal_scores[improved] = positions[improved], scores[improved]
        best = int(np.argmax(personal_scores))
        global_best = personal[best].copy()
        trace.append(float(personal_scores[best]))
        if generation + 1 < generations:
            # Inertia 0.7 mempertahankan sebagian arah. Bobot 1.5 menyeimbangkan pengalaman pribadi dan swarm.
            velocity = (0.7 * velocity
                        + 1.5 * rng.random(positions.shape) * (personal - positions)
                        + 1.5 * rng.random(positions.shape) * (global_best - positions))
            # Batasi langkah dan posisi agar pencarian tetap berada dalam rentang hyperparameter.
            velocity = np.clip(velocity, -0.25 * (upper - lower), 0.25 * (upper - lower))
            positions = np.clip(positions + velocity, lower, upper)
    return global_best, np.asarray(trace)

# Uji independen pada fungsi cekung dengan optimum diketahui di (0.3, -0.4).
def check_swarm():
    visited = []
    target = np.array([0.3, -0.4])
    def objective(point):
        assert ((point >= -1) & (point <= 1)).all()
        visited.append(point)
        return -float(np.sum((point - target) ** 2))
    args = dict(bounds=[[-1, 1], [-1, 1]], initial=[0, 0], particles=8, generations=30)
    best, history = particle_swarm(objective, **args)
    assert len(visited) == 240 and np.all(np.diff(history) >= 0)
    np.testing.assert_allclose(best, target, atol=0.02)
    repeated, _ = particle_swarm(objective, **args)
    np.testing.assert_array_equal(best, repeated)
check_swarm()
print('PSO: batas posisi, anggaran, perbaikan fitness, optimum uji, dan seed lolos.')

In [ ]:
def search_bonus(y_dev, folds, prepared_folds):
    BASE_PARAMS = dict(kernel='gaussian', **COMMON_PARAMS)
    # Nilai tiap kandidat PSO dengan fold dan preprocessing yang sama.
    # Solver yang gagal atau belum konvergen menghentikan evaluasi kandidat.
    evaluations = []
    def evaluate(position):
        # Skala log menjadikan perpindahan partikel sebagai perubahan rasio C dan gamma.
        C, gamma_factor = 10.0 ** np.asarray(position)
        oof = np.full(DEV_SIZE, np.nan)
        started = time.perf_counter()
        for tr, va, X_tr, X_va in prepared_folds:
            # Faktor 1 setara gamma='scale' pada train fold. Holdout tidak ikut menghitung varians.
            gamma = gamma_factor / (X_tr.shape[1] * X_tr.var())
            with warnings.catch_warnings(record=True) as messages:
                warnings.simplefilter('always', RuntimeWarning)
                model = SVM(**BASE_PARAMS, C=C, gamma=gamma).fit(X_tr, y_dev[tr])
            assert model.converged_ and not messages, [str(m.message) for m in messages]
            oof[va] = model.predict_proba(X_va)
        assert np.isfinite(oof).all() and ((oof >= 0) & (oof <= 1)).all()
        threshold, score = best_threshold(y_dev, oof)
        assert np.isclose(score, f1_score(y_dev, oof >= threshold, average='macro'))
        evaluations.append(dict(evaluation=len(evaluations) + 1, C=float(C),
                                gamma_factor=float(gamma_factor), threshold=threshold,
                                oof_macro_f1=score,
                                fold_std=float(fold_scores(y_dev, oof, folds, threshold).std()),
                                seconds=time.perf_counter() - started))
        print(f'Kandidat {len(evaluations):02d}/{PARTICLES * GENERATIONS}: '
              f'C={C:.3g}, faktor gamma={gamma_factor:.3g}, OOF={score:.4f}', flush=True)
        return score

    # Rentang C 0.01–10 dan faktor gamma 0.05–5 mencakup penalti serta lebar kernel di sekitar baseline.
    started = time.perf_counter()
    with threadpool_limits(limits=1):
        best_position, history = particle_swarm(
            evaluate, bounds=[[-2, 1], [np.log10(0.05), np.log10(5)]], initial=[0, 0],
            particles=PARTICLES, generations=GENERATIONS, seed=SEED)
    search_seconds = time.perf_counter() - started
    evaluations = pd.DataFrame(evaluations)
    # Ambil baseline awal dan kandidat pemenang sesuai posisi terbaik PSO.
    baseline = evaluations.iloc[0].to_dict()
    matches = np.isclose(evaluations['C'], 10.0 ** best_position[0]) & np.isclose(
        evaluations['gamma_factor'], 10.0 ** best_position[1])
    best = evaluations.loc[matches].iloc[0].to_dict()
    np.testing.assert_allclose(10.0 ** best_position, [best['C'], best['gamma_factor']])
    assert len(evaluations) == PARTICLES * GENERATIONS
    assert history[-1] >= baseline['oof_macro_f1'] and np.all(np.diff(history) >= 0)
    display(evaluations.nlargest(5, 'oof_macro_f1').round(4))
    print(f'Pencarian selesai: {search_seconds:.1f} detik.')
    return evaluations, baseline, best, history, search_seconds

In [ ]:
bonus_evaluations, bonus_baseline, bonus_best, bonus_history, bonus_seconds = search_bonus(
    bonus_y, bonus_folds, bonus_prepared)

### Evaluasi Holdout PSO

In [ ]:
def compare_bonus(dev, holdout, y_dev, baseline, best):
    BASE_PARAMS = dict(kernel='gaussian', **COMMON_PARAMS)
    # Latih ulang baseline dan pemenang pada data pengembangan, lalu nilai holdout.
    # Threshold tetap berasal dari OOF. Hasil holdout tidak digunakan untuk memilih kandidat baru.
    prep_final = SVMPreprocessor()
    X_dev = prep_final.fit_transform(dev).to_numpy()
    X_holdout = prep_final.transform(holdout).to_numpy()
    y_holdout = holdout[TARGET].to_numpy()
    comparison_rows = []
    with threadpool_limits(limits=1):
        for name, candidate in [('Baseline Gaussian', baseline), ('Gaussian + PSO', best)]:
            gamma = candidate['gamma_factor'] / (X_dev.shape[1] * X_dev.var())
            started = time.perf_counter()
            with warnings.catch_warnings(record=True) as messages:
                warnings.simplefilter('always', RuntimeWarning)
                model = SVM(**BASE_PARAMS, C=candidate['C'], gamma=gamma).fit(X_dev, y_dev)
            assert model.converged_ and not messages, [str(m.message) for m in messages]
            pred = model.predict(X_holdout, threshold=candidate['threshold'])
            score = macro_f1(y_holdout, pred)
            assert np.isclose(score, f1_score(y_holdout, pred, average='macro'))
            comparison_rows.append(dict(model=name, C=candidate['C'], gamma_factor=candidate['gamma_factor'],
                                        threshold=candidate['threshold'], oof_macro_f1=candidate['oof_macro_f1'],
                                        fold_std=candidate['fold_std'], holdout_macro_f1=score,
                                        fit_seconds=time.perf_counter() - started))
    comparison = pd.DataFrame(comparison_rows).set_index('model')
    display(comparison.round(4))
    return comparison

In [ ]:
bonus_comparison = compare_bonus(bonus_dev, bonus_holdout, bonus_y, bonus_baseline, bonus_best)

### Hasil dan Analisis PSO

In [ ]:
def report_bonus(comparison, evaluations, baseline, best, history, search_seconds, dev, holdout, visual_results):
    # Plot riwayat fitness serta perbandingan skor seleksi dengan holdout.
    fig = pso_plot(history, comparison, DEV_SIZE, HOLDOUT_SIZE)
    show_plot(fig, FIG_DIR / (EXP_NAME + '_pso.png'))

    # Jelaskan hasil yang teramati, termasuk jika peningkatan OOF tidak bertahan di holdout.
    oof_gain = comparison.iloc[1]['oof_macro_f1'] - comparison.iloc[0]['oof_macro_f1']
    holdout_gain = comparison.iloc[1]['holdout_macro_f1'] - comparison.iloc[0]['holdout_macro_f1']
    holdout_result = 'meningkat' if holdout_gain > 0 else ('tetap' if holdout_gain == 0 else 'menurun')
    display(Markdown(
        f"PSO memilih C={best['C']:.4g}, faktor gamma={best['gamma_factor']:.4g}, "
        f"dan threshold={best['threshold']:.2f}. Macro F1 OOF berubah dari "
        f"{baseline['oof_macro_f1']:.4f} menjadi {best['oof_macro_f1']:.4f}, "
        f"dengan selisih {oof_gain:+.4f}. Skor ini digunakan untuk memilih kandidat. "
        f"\n\nPada holdout terpisah, Macro F1 {holdout_result} dari "
        f"**{comparison.iloc[0]['holdout_macro_f1']:.4f}** menjadi "
        f"**{comparison.iloc[1]['holdout_macro_f1']:.4f}** (selisih {holdout_gain:+.4f}). "
        f"Pencarian memakai {len(evaluations)} evaluasi dengan waktu {search_seconds:.1f} detik. "
        "\n\nEksperimen dengan satu seed dan holdout berukuran kecil belum cukup untuk "
        "menyimpulkan bahwa konfigurasi terpilih lebih baik secara umum atau meningkatkan "
        "skor Kaggle. Jumlah evaluasi yang terbatas juga belum membuktikan optimum global."
    ))
    # Simpan kandidat, metrik, hash data, dan ID pembagian agar eksperimen dapat ditelusuri.
    evaluations.to_csv(RUN_DIR / 'evaluations.csv', index=False)
    comparison.to_csv(RUN_DIR / 'bonus_comparison.csv')
    summary = dict(seed=SEED, dev_rows=DEV_SIZE, holdout_rows=HOLDOUT_SIZE,
                   particles=PARTICLES, generations=GENERATIONS, search_seconds=search_seconds,
                   train_sha256=hashlib.sha256((DATA_DIR / 'train.csv').read_bytes()).hexdigest(),
                   dev_ids=dev['id'].tolist(), holdout_ids=holdout['id'].tolist(),
                   visualizations=visual_results, history=history.tolist(),
                   comparison=comparison.reset_index().to_dict(orient='records'))
    (RUN_DIR / 'bonus_results.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print('Gambar:', (FIG_DIR / (EXP_NAME + '_boundaries.png')).relative_to(ROOT),
          'dan', (FIG_DIR / (EXP_NAME + '_pso.png')).relative_to(ROOT))

In [ ]:
report_bonus(bonus_comparison, bonus_evaluations, bonus_baseline, bonus_best, bonus_history,
             bonus_seconds, bonus_dev, bonus_holdout, bonus_visual)

#### Insights

PSO memilih C dan faktor gamma berdasarkan Macro F1 OOF pada data pengembangan.
Threshold berasal dari OOF tersebut dan tetap digunakan saat evaluasi holdout.
Kenaikan fitness pencarian belum menjamin kenaikan skor holdout atau skor Kaggle.
Hasil aktual dan perbandingannya ditampilkan oleh sel bonus, termasuk jika holdout menurun.

---

# Referensi <a name="13"></a>

---

- Optimasi blok QP memakai [SciPy SLSQP](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-slsqp.html).
SLSQP merupakan solver numerik untuk objective kuadratik dengan kendala equality dan bounds.
- Kalibrasi sigmoid memakai [SciPy BFGS](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-bfgs.html).
BFGS memperbarui parameter sigmoid melalui pendekatan matriks kelengkungan.
- J. Kennedy dan R. Eberhart, *Particle Swarm Optimization*, ICNN, 1995.
[DOI 10.1109/ICNN.1995.488968](https://doi.org/10.1109/ICNN.1995.488968).
- Y. Shi dan R. Eberhart, *A Modified Particle Swarm Optimizer*, IEEE ICEC, 1998.
[DOI 10.1109/ICEC.1998.699146](https://doi.org/10.1109/ICEC.1998.699146).
- S.-W. Lin, K.-C. Ying, S.-C. Chen, dan Z.-J. Lee,
*Particle swarm optimization for parameter determination and feature selection of support vector machines*,
Expert Systems with Applications, 2008.
[DOI 10.1016/j.eswa.2007.08.088](https://doi.org/10.1016/j.eswa.2007.08.088).
Notebook ini mengadaptasi pencarian parameter SVM dan tidak menerapkan feature selection pada penelitian tersebut.

In [ ]:
run_summary = dict(experiment=EXP_NAME, smoke_test=SMOKE_TEST, seed=SEED,
                   train_rows=len(train_clean), test_rows=len(test_raw), parameters=PARAMS,
                   solver='SLSQP', calibration='BFGS',
                   comparison=comparison.reset_index().to_dict(orient='records'),
                   historical_oof=HISTORICAL_OOF, historical_threshold=HISTORICAL_THRESHOLD,
                   historical_public_score=PUBLIC_SCORE, historical_replay_verified=replay_verified,
                   model_path=str(MODEL_PATH.relative_to(ROOT)),
                   csv_path=str(RERUN_SUB_PATH.relative_to(ROOT)))
(RUN_DIR / 'run_summary.json').write_text(json.dumps(run_summary, indent=2), encoding='utf-8')
print('Ringkasan hasil:', RUN_DIR / 'run_summary.json')